# Notebook 19 — RL for Tool Use and Structured Outputs

One of the best modern uses of RL-style post-training is **not** vague open-ended helpfulness. It is precise behavior that can be verified: valid JSON, correct schema fields, correct tool choice, and correct arguments. This notebook shows how to turn those checks into inspectable rewards.

## What you will build

- a tiny benchmark for tool calls and structured outputs
- schema validators and argument checkers
- combined rewards for strict JSON plus tool correctness
- group-relative scoring on tool-use rollouts
- toy training logs that show adherence and correctness rates

## Design principle

If you can write a parser and a verifier, you can often train toward the behavior more safely than by relying on vague preference labels alone. That does **not** mean RL is universal. It means RL becomes attractive when success is concrete.

In [ ]:
# --- Google Colab Pro Fine-Tuning + Evaluation Setup ---
%%capture
!pip install -q --upgrade pip
!pip install -q unsloth datasets trl peft accelerate bitsandbytes pandas matplotlib scikit-learn

import json
import math
import random
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from datasets import Dataset, DatasetDict, load_dataset
from google.colab import drive
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

drive.mount('/content/drive')

CACHE_DIR = "/content/drive/MyDrive/models"
OUTPUT_DIR = "/content/drive/MyDrive/castalia-finetuning"
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
DTYPE = None
LOAD_IN_4BIT = True

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    cache_dir=CACHE_DIR,
)

tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

def formatting_prompts_func(batch):
    return {"text": batch["text"]}

print("✓ Fine-tuning stack ready")
print("  Model:", MODEL_NAME)
print("  CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("  BF16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

## Step 1: Add notebook helpers and artifact paths

In [ ]:
from collections import Counter, defaultdict
import statistics
import re

random.seed(19)

ARTIFACT_DIR = Path("artifacts") / "notebook_19_tool_use_structured_outputs"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def to_markdown_table(rows, columns):
    header = "| " + " | ".join(columns) + " |"
    divider = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = []
    for row in rows:
        body.append("| " + " | ".join(str(row.get(column, "")) for column in columns) + " |")
    return "\n".join([header, divider, *body])

def softmax_dict(weights):
    largest = max(weights.values())
    exps = {key: math.exp(value - largest) for key, value in weights.items()}
    total = sum(exps.values())
    return {key: value / total for key, value in exps.items()}

def sample_from_probs(probs):
    draw = random.random()
    cumulative = 0.0
    for key, value in probs.items():
        cumulative += value
        if draw <= cumulative:
            return key
    return next(reversed(probs))

print("Artifact directory:", ARTIFACT_DIR.resolve())

## Step 2: Define tool schemas and benchmark tasks

Every task names the correct tool and the correct arguments. That gives us exact supervision without needing human judges.

In [ ]:
tool_schemas = {
    "search_docs": {
        "required": {"query": str, "top_k": int},
        "optional": {},
    },
    "route_ticket": {
        "required": {"queue": str, "priority": str, "summary": str},
        "optional": {},
        "enums": {"priority": {"low", "medium", "high"}},
    },
    "calculator": {
        "required": {"expression": str},
        "optional": {},
    },
}

tool_tasks = [
    {
        "id": "docs_password_reset",
        "prompt": "Search the docs for password reset instructions. Respond with strict JSON using tool and args keys.",
        "expected_tool": "search_docs",
        "expected_args": {"query": "password reset instructions", "top_k": 3},
    },
    {
        "id": "route_incident",
        "prompt": "Send a login outage to the identity queue at high priority. Respond with strict JSON using tool and args keys.",
        "expected_tool": "route_ticket",
        "expected_args": {"queue": "identity", "priority": "high", "summary": "login outage"},
    },
    {
        "id": "calc_tax",
        "prompt": "Use the calculator to compute 18 * 1.2. Respond with strict JSON using tool and args keys.",
        "expected_tool": "calculator",
        "expected_args": {"expression": "18 * 1.2"},
    },
]

print(to_markdown_table(tool_tasks, ["id", "expected_tool", "expected_args"]))

## Step 3: Build candidate output styles

We simulate several habits a model might learn:

- strict JSON and usually correct
- correct content but extra prose
- wrong tool choice
- wrong argument types or values
- schema drift with extra or missing keys

In [ ]:
STYLE_BEHAVIOR = {
    "strict_json": {"tool_prob": 0.92, "args_prob": 0.9, "strict_prob": 0.98},
    "extra_prose": {"tool_prob": 0.88, "args_prob": 0.86, "strict_prob": 0.15},
    "wrong_tool": {"tool_prob": 0.15, "args_prob": 0.7, "strict_prob": 0.92},
    "bad_args": {"tool_prob": 0.9, "args_prob": 0.2, "strict_prob": 0.95},
    "schema_drift": {"tool_prob": 0.82, "args_prob": 0.55, "strict_prob": 0.55},
}

style_weights = {style: 0.0 for style in STYLE_BEHAVIOR}
style_weights

## Step 4: Render structured-output candidates

In [ ]:
def mutate_args(task):
    args = dict(task["expected_args"])
    if task["expected_tool"] == "search_docs":
        args["top_k"] = "3"
    elif task["expected_tool"] == "route_ticket":
        args["priority"] = "urgent"
    elif task["expected_tool"] == "calculator":
        args["expression"] = 21.6
    return args

def alternate_tool(task):
    for tool_name in tool_schemas:
        if tool_name != task["expected_tool"]:
            return tool_name
    return task["expected_tool"]

def render_candidate(task, style):
    behavior = STYLE_BEHAVIOR[style]
    tool_name = task["expected_tool"] if random.random() < behavior["tool_prob"] else alternate_tool(task)
    args = dict(task["expected_args"]) if random.random() < behavior["args_prob"] else mutate_args(task)

    payload = {"tool": tool_name, "args": args}
    if style == "schema_drift":
        payload["confidence"] = 0.91
    if style == "bad_args" and "args" in payload and random.random() < 0.5:
        payload["args"]["unexpected"] = True

    strict = random.random() < behavior["strict_prob"]
    json_text = json.dumps(payload, ensure_ascii=False)
    if strict:
        text = json_text
    else:
        text = f"Sure — here is the tool call: {json_text}"

    return {
        "task_id": task["id"],
        "style": style,
        "text": text,
        "word_count": len(text.split()),
    }

demo_candidates = [render_candidate(tool_tasks[0], style) for style in STYLE_BEHAVIOR]
pd.DataFrame(demo_candidates)

## Step 5: Validate strict JSON and schema adherence

For structured outputs, parsing success is itself a major part of the reward.

In [ ]:
def parse_strict_json(text):
    stripped = text.strip()
    if not stripped.startswith("{") or not stripped.endswith("}"):
        return None, "non_json_wrapper"
    try:
        return json.loads(stripped), None
    except json.JSONDecodeError:
        return None, "json_decode_error"

def validate_schema(payload):
    if not isinstance(payload, dict):
        return {"schema_ok": False, "reason": "payload_not_object"}
    if set(payload.keys()) != {"tool", "args"}:
        return {"schema_ok": False, "reason": "top_level_keys"}
    if not isinstance(payload["tool"], str):
        return {"schema_ok": False, "reason": "tool_not_string"}
    if not isinstance(payload["args"], dict):
        return {"schema_ok": False, "reason": "args_not_object"}
    return {"schema_ok": True, "reason": "ok"}

validation_rows = []
for candidate in demo_candidates:
    payload, parse_error = parse_strict_json(candidate["text"])
    schema = validate_schema(payload) if payload is not None else {"schema_ok": False, "reason": parse_error}
    validation_rows.append({
        "style": candidate["style"],
        "parse_error": parse_error or "",
        "schema_ok": schema["schema_ok"],
        "reason": schema["reason"],
        "text": candidate["text"],
    })

pd.DataFrame(validation_rows)

## Step 6: Check tool identity and argument correctness

Schema correctness is necessary but not sufficient. A perfectly valid JSON object can still call the wrong tool or pass the wrong arguments.

In [ ]:
def validate_arguments(tool_name, args):
    schema = tool_schemas.get(tool_name)
    if schema is None:
        return {"args_ok": False, "reason": "unknown_tool"}
    required = schema["required"]
    enums = schema.get("enums", {})

    if set(args.keys()) != set(required.keys()):
        return {"args_ok": False, "reason": "arg_keys"}
    for key, expected_type in required.items():
        if not isinstance(args.get(key), expected_type):
            return {"args_ok": False, "reason": f"type_{key}"}
    for key, allowed_values in enums.items():
        if args.get(key) not in allowed_values:
            return {"args_ok": False, "reason": f"enum_{key}"}
    return {"args_ok": True, "reason": "ok"}

def exact_tool_call_match(task, payload):
    if payload["tool"] != task["expected_tool"]:
        return False
    return payload["args"] == task["expected_args"]

tool_check_rows = []
for candidate in demo_candidates:
    payload, parse_error = parse_strict_json(candidate["text"])
    if payload is None:
        tool_check_rows.append({
            "style": candidate["style"],
            "tool_ok": False,
            "args_ok": False,
            "exact_match": False,
            "reason": parse_error,
        })
        continue
    args_status = validate_arguments(payload["tool"], payload["args"])
    tool_check_rows.append({
        "style": candidate["style"],
        "tool_ok": payload["tool"] == tool_tasks[0]["expected_tool"],
        "args_ok": args_status["args_ok"],
        "exact_match": exact_tool_call_match(tool_tasks[0], payload) if args_status["args_ok"] else False,
        "reason": args_status["reason"],
    })

pd.DataFrame(tool_check_rows)

## Step 7: Combine the reward components

We want the reward to reflect the actual deployment goal:

- parse cleanly
- match the schema exactly
- call the correct tool
- pass correct arguments
- avoid extra prose around the JSON

In [ ]:
def score_candidate(task, candidate, length_penalty_weight=0.01):
    payload, parse_error = parse_strict_json(candidate["text"])
    if payload is None:
        return {
            "parse_reward": -0.4,
            "schema_reward": -0.3,
            "tool_reward": 0.0,
            "args_reward": 0.0,
            "constraint_bonus": 0.0,
            "length_penalty": round(-length_penalty_weight * max(0, candidate["word_count"] - 18), 3),
            "total_reward": round(-0.7 - length_penalty_weight * max(0, candidate["word_count"] - 18), 3),
        }

    schema_status = validate_schema(payload)
    args_status = validate_arguments(payload["tool"], payload["args"])
    parse_reward = 0.3
    schema_reward = 0.25 if schema_status["schema_ok"] else -0.25
    tool_reward = 0.35 if payload["tool"] == task["expected_tool"] else -0.2
    args_reward = 0.35 if args_status["args_ok"] and payload["args"] == task["expected_args"] else -0.15
    constraint_bonus = 0.15 if candidate["text"].strip().startswith("{") else 0.0
    length_penalty = round(-length_penalty_weight * max(0, candidate["word_count"] - 18), 3)
    total = parse_reward + schema_reward + tool_reward + args_reward + constraint_bonus + length_penalty
    return {
        "parse_reward": round(parse_reward, 3),
        "schema_reward": round(schema_reward, 3),
        "tool_reward": round(tool_reward, 3),
        "args_reward": round(args_reward, 3),
        "constraint_bonus": round(constraint_bonus, 3),
        "length_penalty": length_penalty,
        "total_reward": round(total, 3),
    }

scored_demo = []
for candidate in demo_candidates:
    scored_demo.append({**candidate, **score_candidate(tool_tasks[0], candidate)})

pd.DataFrame(scored_demo)[["style", "parse_reward", "schema_reward", "tool_reward", "args_reward", "constraint_bonus", "total_reward", "text"]]

## Step 8: Inspect constrained-output examples

The difference between "valid enough" and "deployment-safe" often comes from strictness at the edges.

In [ ]:
constraint_examples = [
    {"label": "perfect", "text": json.dumps({"tool": "search_docs", "args": {"query": "password reset instructions", "top_k": 3}})},
    {"label": "extra_prose", "text": "Use this call: " + json.dumps({"tool": "search_docs", "args": {"query": "password reset instructions", "top_k": 3}})},
    {"label": "wrong_top_k_type", "text": json.dumps({"tool": "search_docs", "args": {"query": "password reset instructions", "top_k": "3"}})},
    {"label": "extra_key", "text": json.dumps({"tool": "search_docs", "args": {"query": "password reset instructions", "top_k": 3}, "confidence": 0.8})},
]

constraint_rows = []
for example in constraint_examples:
    candidate = {"task_id": tool_tasks[0]["id"], "style": example["label"], "text": example["text"], "word_count": len(example["text"].split())}
    constraint_rows.append({"label": example["label"], **score_candidate(tool_tasks[0], candidate)})

pd.DataFrame(constraint_rows)

## Step 9: Sample one rollout group for a tool task

This makes the group-relative part of GRPO visible on structured-output behavior.

In [ ]:
def relative_advantages(values):
    mean_value = statistics.mean(values)
    stdev = statistics.pstdev(values)
    if stdev == 0:
        return [0.0 for _ in values]
    return [round((value - mean_value) / stdev, 3) for value in values]

def sample_group(task, weights, group_size=5, length_penalty_weight=0.01):
    probs = softmax_dict(weights)
    group = []
    for _ in range(group_size):
        style = sample_from_probs(probs)
        candidate = render_candidate(task, style)
        scored = {**candidate, **score_candidate(task, candidate, length_penalty_weight=length_penalty_weight)}
        group.append(scored)
    advantages = relative_advantages([item["total_reward"] for item in group])
    for item, advantage in zip(group, advantages):
        item["advantage"] = advantage
    return group

example_group = sample_group(tool_tasks[1], style_weights, group_size=6)
pd.DataFrame(example_group)[["style", "total_reward", "advantage", "word_count", "text"]]

## Step 10: Run a toy GRPO loop for structured outputs

Again, this is a teaching loop. It shows how a policy can shift toward strict schema-correct tool calls when rewards are concrete.

In [ ]:
def train_tool_policy(tasks, steps=100, group_size=5, lr=0.2, length_penalty_weight=0.01, seed=19):
    random.seed(seed)
    weights = {style: 0.0 for style in STYLE_BEHAVIOR}
    history = []

    for step in range(1, steps + 1):
        task = random.choice(tasks)
        group = sample_group(task, weights, group_size=group_size, length_penalty_weight=length_penalty_weight)
        for item in group:
            weights[item["style"]] += lr * item["advantage"]

        parse_rate = statistics.mean(1.0 if item["parse_reward"] > 0 else 0.0 for item in group)
        schema_rate = statistics.mean(1.0 if item["schema_reward"] > 0 else 0.0 for item in group)
        tool_rate = statistics.mean(1.0 if item["tool_reward"] > 0 else 0.0 for item in group)
        args_rate = statistics.mean(1.0 if item["args_reward"] > 0 else 0.0 for item in group)

        history.append({
            "step": step,
            "task_id": task["id"],
            "mean_reward": round(statistics.mean(item["total_reward"] for item in group), 3),
            "parse_rate": round(parse_rate, 3),
            "schema_rate": round(schema_rate, 3),
            "tool_rate": round(tool_rate, 3),
            "args_rate": round(args_rate, 3),
            "avg_words": round(statistics.mean(item["word_count"] for item in group), 2),
            "winner_style": max(group, key=lambda item: item["total_reward"])["style"],
        })

    return weights, pd.DataFrame(history)

final_weights, history_df = train_tool_policy(tool_tasks)
history_df.head()

## Step 11: Plot adherence and correctness over time

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

history_df["reward_smooth"] = history_df["mean_reward"].rolling(10, min_periods=1).mean()
history_df["schema_smooth"] = history_df["schema_rate"].rolling(10, min_periods=1).mean()
history_df["args_smooth"] = history_df["args_rate"].rolling(10, min_periods=1).mean()

axes[0].plot(history_df["step"], history_df["reward_smooth"])
axes[0].set_title("Mean reward")
axes[0].set_xlabel("step")

axes[1].plot(history_df["step"], history_df["schema_smooth"], color="orange")
axes[1].set_title("Schema adherence")
axes[1].set_xlabel("step")

axes[2].plot(history_df["step"], history_df["args_smooth"], color="green")
axes[2].set_title("Argument correctness")
axes[2].set_xlabel("step")

plt.tight_layout()
plt.show()

## Step 12: Inspect the learned style distribution

In [ ]:
style_probabilities = softmax_dict(final_weights)
pd.DataFrame(
    [{"style": style, "probability": round(prob, 3)} for style, prob in style_probabilities.items()]
).sort_values("probability", ascending=False)

## Step 13: Review failure buckets

A useful RL run should reduce failure buckets you care about in deployment.

In [ ]:
failure_buckets = defaultdict(int)
for task in tool_tasks:
    for _ in range(40):
        candidate = render_candidate(task, sample_from_probs(style_probabilities))
        payload, parse_error = parse_strict_json(candidate["text"])
        if payload is None:
            failure_buckets[f"parse::{parse_error}"] += 1
            continue
        schema_status = validate_schema(payload)
        if not schema_status["schema_ok"]:
            failure_buckets[f"schema::{schema_status['reason']}"] += 1
            continue
        args_status = validate_arguments(payload["tool"], payload["args"])
        if payload["tool"] != task["expected_tool"]:
            failure_buckets["tool::wrong_tool"] += 1
        elif not args_status["args_ok"] or payload["args"] != task["expected_args"]:
            failure_buckets[f"args::{args_status['reason']}"] += 1
        else:
            failure_buckets["success"] += 1

bucket_df = pd.DataFrame(
    [{"bucket": bucket, "count": count} for bucket, count in sorted(failure_buckets.items())]
).sort_values("count", ascending=False)
bucket_df

## Step 14: Prepare training-ready artifacts

As with reasoning RL, the implementation details of the trainer may move. The durable part is the dataset plus the reward contract.

In [ ]:
training_records = [
    {
        "prompt": task["prompt"],
        "expected_tool": task["expected_tool"],
        "expected_args": task["expected_args"],
    }
    for task in tool_tasks
]

training_dataset = Dataset.from_list(training_records)
reward_contract = {
    "components": ["parse_reward", "schema_reward", "tool_reward", "args_reward", "constraint_bonus", "length_penalty"],
    "goal": "strict JSON, correct tool, correct arguments, no extra prose",
    "notes": "Constrained decoding can help too, but reward design still matters for borderline cases.",
}

print(training_dataset)
print(json.dumps(reward_contract, indent=2))

## Step 15: Save benchmark and log artifacts

In [ ]:
history_path = ARTIFACT_DIR / "tool_policy_history.csv"
bucket_path = ARTIFACT_DIR / "failure_buckets.csv"
contract_path = ARTIFACT_DIR / "reward_contract.json"
benchmark_path = ARTIFACT_DIR / "tool_tasks.json"

history_df.to_csv(history_path, index=False)
bucket_df.to_csv(bucket_path, index=False)
with open(contract_path, "w", encoding="utf-8") as handle:
    json.dump(reward_contract, handle, indent=2)
with open(benchmark_path, "w", encoding="utf-8") as handle:
    json.dump(tool_tasks, handle, indent=2)

print("Saved:", history_path)
print("Saved:", bucket_path)
print("Saved:", contract_path)
print("Saved:", benchmark_path)

## Recap

You now have a concrete pattern for RL on structured outputs:

- reward strict parsing, not just approximate readability
- validate schema adherence separately from semantic correctness
- check tool choice and arguments independently
- use group-relative rewards to reinforce the best rollout for each prompt
- keep the task small and verifiable so reward design stays inspectable